In [63]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [64]:
data = pd.read_csv('Churn_Modelling.csv')

In [65]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [66]:
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [67]:
label_encoder_gender =  LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [68]:
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df =  pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [69]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [70]:
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

In [71]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [72]:
x = data.drop('EstimatedSalary',axis=1)
y = data['EstimatedSalary']

In [73]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [74]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [75]:
## Save the Encoder and scaler
with open('label_encoder_gender.pkl','wb') as file:
  pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_gro.pkl','wb') as file:
  pickle.dump(onehot_encoder_geo,file)

## Save in pickel
with open('scaler.pkl','wb') as file:
  pickle.dump(scaler,file)

In [76]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [77]:
model = Sequential([
    Dense(138 ,activation='relu',input_shape=(x_train.shape[1],)),
    Dense(500, activation='relu'),
    Dense(1)
])



s:\Mechine Learning\sec_53\ANN\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [78]:
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

In [79]:
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 138)            │         1,794 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 500)            │        69,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │           501 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71,795 (280.45 KB)

 Trainable params: 71,795 (280.45 KB)

 Non-trainable params: 0 (0.00 B)

In [80]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime


log_dir = 'reglog/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [81]:
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [82]:
history = model.fit(
    x_train , y_train,
    validation_data =(x_test , y_test),
    epochs = 100,
    callbacks = [early_stopping_callback,tensorboard_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 98924.8125 - mae: 98924.8125 - val_loss: 92476.3906 - val_mae: 92476.3906
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 80014.1719 - mae: 80014.1719 - val_loss: 62873.4258 - val_mae: 62873.4258
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 54925.0938 - mae: 54925.0938 - val_loss: 50950.8438 - val_mae: 50950.8438
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 50342.9219 - mae: 50342.9219 - val_loss: 50492.7656 - val_mae: 50492.7656
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 50027.4453 - mae: 50027.4453 - val_loss: 50454.0312 - val_mae: 50454.0312
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49924.1914 - mae: 49924.1914 - val_loss: 50389.7539 - val_mae: 50389.7539
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 49836.7539 - mae: 49836.7539 - val_loss: 50348.8594 - val_mae: 50348.8594
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - 

In [83]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [84]:
%tensorboard --logdir reglog/fit

Reusing TensorBoard on port 6006 (pid 4912), started 0:10:04 ago. (Use '!kill 4912' to kill it.)

In [85]:
test_loss,test_mae = model.evaluate(x_test,y_test)
print(test_mae)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 50257.0469 - mae: 50257.0469
50257.046875


In [86]:
model.save('regression_model.h5')